# День 2 — hidden states и CLS-эмбеддинги

Модель та же, что в день 1: `distilbert-base-uncased`. Тексты токенизируются функцией `tokenize_texts`.

`last_hidden_state` имеет форму `[batch, длина последовательности, 768]`. Эмбеддинг всего текста — вектор первого токена `[CLS]`.

## Задача 1. Загрузка модели

In [1]:
import sys
from pathlib import Path

import torch
from transformers import AutoModel

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.embeddings import get_embeddings, load_model, similarity
from src.tokenization import MODEL_NAME, load_tokenizer

tokenizer = load_tokenizer(MODEL_NAME)
model = load_model(MODEL_NAME)
print(AutoModel.__name__, model.name_or_path)
print(f"training: {model.training}")
print(model)

W1023 17:47:09.746000 37416 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AutoModel distilbert-base-uncased
training: False
DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): L

## Задача 2. Hidden states для одного текста

Токенизатор возвращает ещё `token_type_ids`. DistilBERT их не использует, поэтому в модель идут только `input_ids` и `attention_mask`.

In [2]:
text = "This movie was absolutely amazing!"
tokens = tokenizer(text, return_tensors="pt")
print(tokens.keys())

with torch.no_grad():
    outputs = model(
        input_ids=tokens["input_ids"],
        attention_mask=tokens["attention_mask"],
    )

print(type(outputs))
print(outputs.last_hidden_state.shape)

cls_embedding = outputs.last_hidden_state[:, 0, :]
print(f"CLS embedding shape: {cls_embedding.shape}")
print(f"CLS embedding: {cls_embedding[0][:5]}...")

KeysView({'input_ids': tensor([[ 101, 2023, 3185, 2001, 7078, 6429,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])})


<class 'transformers.modeling_outputs.BaseModelOutput'>
torch.Size([1, 8, 768])
CLS embedding shape: torch.Size([1, 768])
CLS embedding: tensor([ 0.0682, -0.0685,  0.1918,  0.0139, -0.0544])...


## Задачи 3–4. Эмбеддинги нескольких текстов

`get_embeddings` режет список на батчи, вызывает `tokenize_texts` и забирает CLS-вектор каждого текста.

In [3]:
texts = [
    "This movie was absolutely amazing!",
    "Terrible movie, waste of time.",
    "Pretty good, I liked it.",
    "Boring and too long.",
]

embeddings = get_embeddings(texts, tokenizer, model)
print(f"Embeddings shape: {embeddings.shape}")
print("Ожидается: (4, 768) для DistilBERT")

Embeddings shape: (4, 768)
Ожидается: (4, 768) для DistilBERT


## Задача 5. Сходство текстов

In [4]:
sim1 = similarity("Great movie!", "Amazing film!", tokenizer, model)
sim2 = similarity("Great movie!", "Terrible film!", tokenizer, model)

print(f"Сходство похожих: {sim1:.3f}")
print(f"Сходство разных: {sim2:.3f}")

Сходство похожих: 0.995
Сходство разных: 0.984
